# Riboflavin: train-test splits

Creates the 50 random 50/21 train-test splits (seeds 1000-1049) from `riboflavin_raw_full.csv`; predictors are standardised with training-sample means and SDs. Run this first.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path.cwd().resolve()

DATA_FILE_CANDIDATES = [
    ROOT / "riboflavin_raw_full.csv",
    ROOT / "ribo" / "riboflavin_raw_full.csv",
    ROOT.parent / "ribo" / "riboflavin_raw_full.csv",
]

DATA_FILE = next(
    (path for path in DATA_FILE_CANDIDATES if path.exists()),
    DATA_FILE_CANDIDATES[0]
)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Cannot find riboflavin_raw_full.csv. Tried:\n"
        + "\n".join(str(p) for p in DATA_FILE_CANDIDATES)
    )

N_REPLICATIONS = 50
N_TRAIN = 50
N_TEST = 21

SPLIT_SEEDS = list(range(1000, 1000 + N_REPLICATIONS))

OUTPUT_ROOT = ROOT / "riboflavin_prediction_splits_50x_50train_21test"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(DATA_FILE)

# Remove accidental CSV index columns if present
df = df.loc[
    :,
    ~df.columns.astype(str).str.startswith("Unnamed")
].copy()

if "y" not in df.columns:
    raise ValueError(
        "riboflavin_raw_full.csv must contain a response column named 'y'."
    )

feature_cols = [c for c in df.columns if c != "y"]

n = len(df)
p = len(feature_cols)

print("Raw data file:", DATA_FILE)
print("n =", n)
print("p =", p)

if n != 71:
    raise ValueError(f"Expected n=71, but found n={n}.")

if N_TRAIN + N_TEST != n:
    raise ValueError(
        f"N_TRAIN + N_TEST must equal n. "
        f"Got {N_TRAIN} + {N_TEST} != {n}."
    )

X_raw = df[feature_cols].to_numpy(dtype=np.float64)
y_raw = df["y"].to_numpy(dtype=np.float64)


split_summary = []

for rep, split_seed in enumerate(SPLIT_SEEDS):

    rng = np.random.default_rng(split_seed)

    # Randomly permute the 71 observations
    perm = rng.permutation(n)

    train_idx = perm[:N_TRAIN]
    test_idx = perm[N_TRAIN:]

    if len(train_idx) != 50:
        raise RuntimeError("Training set does not contain 50 observations.")

    if len(test_idx) != 21:
        raise RuntimeError("Test set does not contain 21 observations.")


    X_train_raw = X_raw[train_idx].copy()
    X_test_raw = X_raw[test_idx].copy()

    # y IS NOT centred or standardised
    y_train = y_raw[train_idx].copy()
    y_test = y_raw[test_idx].copy()


    train_means = X_train_raw.mean(axis=0)

    # Population SD (ddof=0)
    train_sds = X_train_raw.std(axis=0, ddof=0)

    # Protect against zero-variance columns in a particular
    # 50-observation training sample.
    zero_sd = train_sds == 0.0
    train_sds_safe = train_sds.copy()
    train_sds_safe[zero_sd] = 1.0

    X_train_std = (
        X_train_raw - train_means
    ) / train_sds_safe

    # Standardise test X with the training mean and training SD
    X_test_std = (
        X_test_raw - train_means
    ) / train_sds_safe


    assert X_train_std.shape == (50, p)
    assert X_test_std.shape == (21, p)

    # Training columns should have approximately mean 0.
    max_abs_train_mean = np.max(
        np.abs(X_train_std.mean(axis=0))
    )

    # --------------------------------------------------------
    # Create replication folder
    # --------------------------------------------------------

    rep_dir = OUTPUT_ROOT / f"rep_{rep:02d}_seed_{split_seed}"
    rep_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Save STANDARDISED training data
    #
    # Format:
    # y, gene1, gene2, ..., gene4088
    # --------------------------------------------------------

    train_df = pd.DataFrame(
        X_train_std,
        columns=feature_cols
    )
    train_df.insert(0, "y", y_train)

    train_file = rep_dir / "train.csv"
    train_df.to_csv(train_file, index=False)

    # --------------------------------------------------------
    # Save STANDARDISED test data
    # --------------------------------------------------------

    test_df = pd.DataFrame(
        X_test_std,
        columns=feature_cols
    )
    test_df.insert(0, "y", y_test)

    test_file = rep_dir / "test.csv"
    test_df.to_csv(test_file, index=False)

    # --------------------------------------------------------
    # Save scaling parameters
    # --------------------------------------------------------

    scaler_df = pd.DataFrame({
        "variable_name": feature_cols,
        "train_mean": train_means,
        "train_sd_original": train_sds,
        "train_sd_used": train_sds_safe,
        "zero_sd_in_training": zero_sd.astype(int),
    })

    scaler_file = rep_dir / "scaler.csv"
    scaler_df.to_csv(scaler_file, index=False)

    # --------------------------------------------------------
    # Save original row indices
    # --------------------------------------------------------

    index_df = pd.concat(
        [
            pd.DataFrame({
                "set": "train",
                "position_in_set": np.arange(N_TRAIN),
                "original_row_index": train_idx,
            }),
            pd.DataFrame({
                "set": "test",
                "position_in_set": np.arange(N_TEST),
                "original_row_index": test_idx,
            }),
        ],
        ignore_index=True,
    )

    index_file = rep_dir / "split_indices.csv"
    index_df.to_csv(index_file, index=False)

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    split_summary.append({
        "replication": rep,
        "split_seed": split_seed,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "p": p,
        "y_train_mean": y_train.mean(),
        "y_test_mean": y_test.mean(),
        "max_abs_standardised_train_X_mean":
            max_abs_train_mean,
        "n_zero_sd_predictors": int(zero_sd.sum()),
        "train_file": str(train_file),
        "test_file": str(test_file),
    })

    print(
        f"rep {rep:02d} | seed={split_seed} | "
        f"train={len(train_idx)} | "
        f"test={len(test_idx)} | "
        f"zero-SD cols={zero_sd.sum()}"
    )


# ============================================================
# SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(split_summary)

summary_file = OUTPUT_ROOT / "split_summary.csv"
summary_df.to_csv(summary_file, index=False)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("Output folder:")
print(OUTPUT_ROOT)

print("\nReplications:", len(summary_df))
print("Training observations per replication:", N_TRAIN)
print("Test observations per replication:", N_TEST)
print("Predictors:", p)

print("\nX:")
print("  standardised separately within each replication")
print("  training mean/SD used")
print("  test transformed using training mean/SD")

print("\ny:")
print("  NOT centred")
print("  NOT standardised")
print("  original values retained")

print("\nSummary saved to:")
print(summary_file)